In [5]:
# ==============================================================================
# 🚀 ALPACA SHOWDOWN: 1.0x CASH vs 2.0x MARGIN (CURATED UNIVERSE)
# Tests the Pure Top 1 Strategy using a hand-picked, highly liquid universe
# ==============================================================================

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
import logging
from kaggle_secrets import UserSecretsClient
import alpaca_trade_api as tradeapi

# Suppress warnings and annoying Alpaca retry logs
warnings.filterwarnings('ignore')
logging.getLogger('urllib3').setLevel(logging.WARNING)
logging.getLogger('alpaca_trade_api').setLevel(logging.WARNING)

# ------------------------------------------------------------------------------
# 1. INITIALIZE ALPACA CLIENT
# ------------------------------------------------------------------------------
print("Authenticating with Alpaca...")
try:
    user_secrets = UserSecretsClient()
    API_KEY = user_secrets.get_secret("APCA_API_KEY_ID")
    SECRET_KEY = user_secrets.get_secret("APCA_API_SECRET_KEY")
except Exception as e:
    print("Error: Could not find Alpaca keys in Kaggle Secrets.")
    API_KEY = "YOUR_API_KEY"       
    SECRET_KEY = "YOUR_SECRET_KEY"

api = tradeapi.REST(API_KEY, SECRET_KEY, base_url='https://paper-api.alpaca.markets', api_version='v2')

# ------------------------------------------------------------------------------
# 2. FETCH UNIVERSE (THE OLD CURATED LIST)
# ------------------------------------------------------------------------------
curated_list = [
    'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA', 'AVGO', 'ORCL', 'ADBE', 
    'CRM', 'NFLX', 'AMD', 'INTC', 'CSCO', 'QCOM', 'TXN', 'AMAT', 'INTU', 'MU',
    'NOW', 'PANW', 'PLTR', 'SNOW', 'CRWD', 'NET', 'DDOG', 'ZS', 'MDB', 'FTNT',
    'BRK.B', 'JPM', 'V', 'MA', 'BAC', 'WFC', 'GS', 'MS', 'AXP', 'SPGI', 
    'LLY', 'UNH', 'JNJ', 'ABBV', 'MRK', 'TMO', 'ABT', 'ISRG', 'DHR', 'PFE',
    'WMT', 'HD', 'PG', 'KO', 'COST', 'PEP', 'MCD', 'NKE', 'TM', 'DIS',
    'XOM', 'CVX', 'COP', 'EOG', 'SLB', 'PSX', 'VLO', 'MPC', 'OXY', 'HAL',
    'SPY', 'QQQ', 'IWM', 'DIA', 'VTI', 'VOO', 'EEM', 'VWO', 'GLD', 'SLV',
    'TLT', 'IEF', 'LQD', 'HYG', 'VNQ', 'IYR', 'XLE', 'XLF', 'XLV', 'XLK',
    'VGT', 'IBB', 'GDX', 'USO', 'UNG', 'ARKK', 'IBIT', 'BITO',
    'TSM', 'ASML', 'NVO', 'SAP', 'BABA', 'PDD', 'SHOP', 'SONY', 'BIDU', 'NIO', 
    'MSTR', 'COIN', 'MARA', 'RIOT', 
    'TQQQ', 'SOXL', 'UPRO', 'URTY', 'FAS', 'TECL', 'NAIL', 'CURE', 'DPST', 'DRN', 'DFEN'
]

print("Validating Curated List against Alpaca's active tradable assets...")
active_assets = api.list_assets(status='active', asset_class='us_equity')
tradable_symbols = [a.symbol for a in active_assets if a.tradable]

# Clean up tickers (e.g., BRK.B to BRK-B or BRK/B depending on Alpaca's format)
universe = []
for t in curated_list:
    if t in tradable_symbols: universe.append(t)
    elif t.replace('.', '-') in tradable_symbols: universe.append(t.replace('.', '-'))

print(f"Testing Universe: {len(universe)} Valid Assets")

# ------------------------------------------------------------------------------
# 3. BULK DOWNLOAD HISTORICAL DATA
# ------------------------------------------------------------------------------
print("Downloading Historical Data from Alpaca (This will take ~5 seconds)...")

end_dt = datetime.today() - timedelta(days=1)
start_dt = end_dt - timedelta(days=365 * 10) # 10 Years (Alpaca Free Limit)

prices = pd.DataFrame()

try:
    bars = api.get_bars(universe, tradeapi.TimeFrame.Day, 
                        start_dt.strftime('%Y-%m-%d'), 
                        end_dt.strftime('%Y-%m-%d'), 
                        adjustment='all',
                        feed='iex').df
    
    if not bars.empty:
        bars = bars.reset_index()
        prices = bars.pivot(index='timestamp', columns='symbol', values='close')
        prices.index = pd.to_datetime(prices.index).tz_convert(None).normalize()
        prices.fillna(method='ffill', inplace=True)
        prices.dropna(axis=1, how='all', inplace=True)
        print(f"✓ Successfully loaded Alpaca data for {len(prices.columns)} assets.")
except Exception as e:
    print(f"Alpaca API Error: {e}")

# ------------------------------------------------------------------------------
# 4. SIMULATION ENGINE
# ------------------------------------------------------------------------------
def run_alpaca_sim(start_date_str, freq='M', leverage=1.0):
    if prices.empty: return 0
    
    sim_start = pd.to_datetime(start_date_str)
    if sim_start < prices.index[0]:
        sim_start = prices.index[0] 
        
    dates = prices.index
    if freq == 'D': rule = 'B'
    elif freq == 'W': rule = 'W-FRI'
    elif freq == 'M': rule = 'BM'
    elif freq == 'Y': rule = 'BY'
    
    rebalance_dates = pd.Series(index=dates, data=1).resample(rule).last().index
    valid_dates = []
    for d in rebalance_dates:
        if d >= sim_start:
            loc = prices.index.get_indexer([d], method='pad')[0]
            if loc != -1: valid_dates.append(prices.index[loc])
                
    valid_dates = sorted(list(set(valid_dates)))
    if len(valid_dates) < 2: return 5000.0 
    
    capital = 5000.0
    portfolio_ticker = None
    
    for i in range(1, len(valid_dates)):
        prev_date = valid_dates[i-1]
        curr_date = valid_dates[i]
        
        # 1. Update Capital with Leverage
        if portfolio_ticker is not None:
            p0 = prices.loc[prev_date, portfolio_ticker]
            p1 = prices.loc[curr_date, portfolio_ticker]
            
            if pd.notna(p0) and pd.notna(p1) and p0 > 0:
                trade_return = (p1 / p0) - 1.0
                capital *= (1.0 + (trade_return * leverage))
                if capital <= 0: return 0.0 # WIPEOUT
                
        # 2. Ranking
        loc = prices.index.get_loc(curr_date)
        if loc < 64: 
            portfolio_ticker = None
            continue
            
        recent = prices.iloc[loc-64:loc+1]
        
        try:
            # Price floor > $5.00 to avoid weird reverse-split data artifacts
            current_prices = recent.iloc[-1]
            valid_assets = current_prices[current_prices > 5.0].index
            
            if len(valid_assets) == 0:
                portfolio_ticker = None
                continue
                
            ret_1m = (recent[valid_assets].iloc[-1] / recent[valid_assets].iloc[-22]) - 1.0
            ret_3m = (recent[valid_assets].iloc[-1] / recent[valid_assets].iloc[-64]) - 1.0
            
            scores = (ret_1m * 0.6) + (ret_3m * 0.4)
            scores = scores.replace([np.inf, -np.inf], np.nan).dropna()
            
            if scores.empty or scores.max() <= 0:
                portfolio_ticker = None # Cash
            else:
                portfolio_ticker = scores.idxmax()
        except:
            portfolio_ticker = None
            
    return capital

# ------------------------------------------------------------------------------
# 5. RUN THE COMPARISON SHOWDOWN
# ------------------------------------------------------------------------------
if not prices.empty:
    cohorts = {
        "25 Years Ago*": (datetime.today() - timedelta(days=365*25)).strftime('%Y-%m-%d'),
        "10 Years Ago": (datetime.today() - timedelta(days=365*10)).strftime('%Y-%m-%d'),
        "5 Years Ago": (datetime.today() - timedelta(days=365*5)).strftime('%Y-%m-%d'),
        "1 Year Ago": (datetime.today() - timedelta(days=365)).strftime('%Y-%m-%d'),
        "6 Months Ago": (datetime.today() - timedelta(days=180)).strftime('%Y-%m-%d'),
        "1 Month Ago": (datetime.today() - timedelta(days=30)).strftime('%Y-%m-%d')
    }

    def print_scoreboard(leverage_val, title):
        print("\n" + "="*95)
        print(f"🛡️ {title} (Leverage: {leverage_val}x)")
        print("="*95)
        print(f"{'Timeframe':<15} | {'Daily Check':<16} | {'Weekly Check':<16} | {'Monthly Check':<16} | {'Yearly Check':<16}")
        print("-" * 95)

        for label, d_str in cohorts.items():
            res_d = run_alpaca_sim(d_str, 'D', leverage_val)
            res_w = run_alpaca_sim(d_str, 'W', leverage_val)
            res_m = run_alpaca_sim(d_str, 'M', leverage_val)
            res_y = run_alpaca_sim(d_str, 'Y', leverage_val)
            
            str_d = f"${res_d:,.2f}" if res_d > 0 else "WIPED OUT ($0)"
            str_w = f"${res_w:,.2f}" if res_w > 0 else "WIPED OUT ($0)"
            str_m = f"${res_m:,.2f}" if res_m > 0 else "WIPED OUT ($0)"
            str_y = f"${res_y:,.2f}" if res_y > 0 else "WIPED OUT ($0)"
            
            # Highlight max
            vals = [res_d, res_w, res_m, res_y]
            max_val = max(vals)
            
            if res_d == max_val and res_d > 0: str_d = f"🏆 {str_d}"
            if res_w == max_val and res_w > 0: str_w = f"🏆 {str_w}"
            if res_m == max_val and res_m > 0: str_m = f"🏆 {str_m}"
            if res_y == max_val and res_y > 0: str_y = f"🏆 {str_y}"
            
            print(f"{label:<15} | {str_d:<16} | {str_w:<16} | {str_m:<16} | {str_y:<16}")
                
        print("-" * 95)

    # Print 1.0x Scoreboard
    print_scoreboard(1.0, "CASH ONLY MODE (Safe)")
    
    # Print 2.0x Scoreboard
    print_scoreboard(2.0, "MAX MARGIN MODE (Aggressive)")
    
    print("\n* Note: Alpaca Free IEX data only goes back to ~2016. '25 Years Ago' starts at the earliest available date.")

Authenticating with Alpaca...
Validating Curated List against Alpaca's active tradable assets...
Testing Universe: 123 Valid Assets
✓ Successfully loaded Alpaca data for 123 assets.

🛡️ CASH ONLY MODE (Safe) (Leverage: 1.0x)
Timeframe       | Daily Check      | Weekly Check     | Monthly Check    | Yearly Check    
-----------------------------------------------------------------------------------------------
25 Years Ago*   | 🏆 $32,166.26     | $20,168.78       | $25,754.28       | $4,958.98       
10 Years Ago    | 🏆 $32,166.26     | $20,168.78       | $25,754.28       | $4,958.98       
5 Years Ago     | $5,352.97        | $6,812.36        | $4,329.55        | 🏆 $15,527.67    
1 Year Ago      | $1,533.88        | $3,853.88        | 🏆 $11,257.48     | $5,000.00       
6 Months Ago    | $2,178.99        | $3,975.19        | 🏆 $8,871.85      | $5,000.00       
1 Month Ago     | $3,690.20        | $4,345.90        | 🏆 $5,000.00      | 🏆 $5,000.00     
-----------------------------------

In [7]:
# ==============================================================================
# 🚀 CELL 2 (FIXED): AI PATTERN RECOGNITION & FEATURE IMPORTANCE
# Uses Alpaca Data to find exactly what numbers correlate with massive wins vs wipeouts
# ==============================================================================

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from sklearn.ensemble import GradientBoostingClassifier
import warnings
from kaggle_secrets import UserSecretsClient
import alpaca_trade_api as tradeapi

warnings.filterwarnings('ignore')

# ------------------------------------------------------------------------------
# 1. INITIALIZE ALPACA CLIENT
# ------------------------------------------------------------------------------
print("Step 1: Authenticating with Alpaca and Downloading Data...")
try:
    user_secrets = UserSecretsClient()
    API_KEY = user_secrets.get_secret("APCA_API_KEY_ID")
    SECRET_KEY = user_secrets.get_secret("APCA_API_SECRET_KEY")
except Exception as e:
    API_KEY = "YOUR_API_KEY"       
    SECRET_KEY = "YOUR_SECRET_KEY"

api = tradeapi.REST(API_KEY, SECRET_KEY, base_url='https://paper-api.alpaca.markets', api_version='v2')

# Use a subset of highly volatile assets to find the strongest mathematical signals
universe = ['TQQQ', 'SOXL', 'NVDA', 'TSLA', 'UPRO', 'MSTR', 'COIN', 'SPY', 'QQQ']

end_dt = datetime.today() - timedelta(days=1)
start_dt = end_dt - timedelta(days=365 * 10) # 10 Years

try:
    bars = api.get_bars(universe, tradeapi.TimeFrame.Day, 
                        start_dt.strftime('%Y-%m-%d'), 
                        end_dt.strftime('%Y-%m-%d'), 
                        adjustment='all',
                        feed='iex').df
    
    if not bars.empty:
        bars = bars.reset_index()
        bars['timestamp'] = pd.to_datetime(bars['timestamp']).dt.tz_convert(None).dt.normalize()
        print(f"✓ Downloaded {len(bars)} rows of historical data.")
    else:
        print("❌ No data returned.")
except Exception as e:
    print(f"Alpaca API Error: {e}")


# ------------------------------------------------------------------------------
# 2. FEATURE ENGINEERING (The "Coordinates")
# ------------------------------------------------------------------------------
print("\nStep 2: Calculating Technical Indicators and Future Outcomes...")

all_features = []

for ticker in universe:
    df = bars[bars['symbol'] == ticker].copy()
    df.set_index('timestamp', inplace=True)
    df.sort_index(inplace=True)
    
    if len(df) < 252: continue
        
    # --- BUILDING THE FEATURES (The "Signs") ---
    # 1. Momentum / Velocity
    df['Ret_1M'] = df['close'].pct_change(21)
    df['Ret_3M'] = df['close'].pct_change(63)
    
    # 2. Volatility (Risk)
    df['Vol_1M'] = df['close'].pct_change().rolling(21).std() * np.sqrt(252)
    
    # 3. Moving Average Distance (The Rubber Band Stretch)
    df['SMA_50'] = df['close'].rolling(50).mean()
    df['Dist_SMA50'] = (df['close'] - df['SMA_50']) / df['SMA_50']
    
    df['SMA_200'] = df['close'].rolling(200).mean()
    df['Dist_SMA200'] = (df['close'] - df['SMA_200']) / df['SMA_200']
    
    # 4. Volume Spikes (Institutional Buying/Selling)
    df['Vol_Spike'] = df['volume'] / df['volume'].rolling(21).mean()
    
    # 5. RSI (Overbought / Oversold)
    delta = df['close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df['RSI_14'] = 100 - (100 / (1 + rs))
    
    # --- DEFINING SUCCESS vs FAILURE (The Target) ---
    # What happened NEXT MONTH (21 trading days later)?
    df['Future_Return_1M'] = df['close'].shift(-21) / df['close'] - 1.0
    
    df.dropna(inplace=True)
    
    # Category 1: Massive Win (> 15% gain in next month)
    # Category 0: Normal / Sideways
    # Category -1: Massive Wipeout (< -15% drop in next month)
    
    conditions = [
        (df['Future_Return_1M'] > 0.15),
        (df['Future_Return_1M'] < -0.15)
    ]
    choices = [1, -1]
    df['Target'] = np.select(conditions, choices, default=0)
    
    # Keep extreme cases to train the AI
    extreme_cases = df[df['Target'] != 0].copy()
    all_features.append(extreme_cases)

# Combine all data
master_df = pd.concat(all_features)
print(f"Total extreme events found: {len(master_df)}")

# ------------------------------------------------------------------------------
# 3. TRAIN AI & EXTRACT IMPORTANCE
# ------------------------------------------------------------------------------
print("\nStep 3: Training AI Gradient Boosting Model to rank importance...")

features = ['Ret_1M', 'Ret_3M', 'Vol_1M', 'Dist_SMA50', 'Dist_SMA200', 'Vol_Spike', 'RSI_14']
X = master_df[features]
y = master_df['Target']

# Train Model
clf = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
clf.fit(X, y)

# Get Importance
importance = clf.feature_importances_
feature_rank = pd.DataFrame({'Feature': features, 'Importance': importance}).sort_values('Importance', ascending=False)

print("\n" + "="*80)
print("🧠 AI FEATURE IMPORTANCE (What actually predicts a Win vs Wipeout)")
print("="*80)
for index, row in feature_rank.iterrows():
    print(f"{row['Feature']:<15} : {row['Importance']:.2%}")

# ------------------------------------------------------------------------------
# 4. EXTRACT THE EXACT COORDINATES
# ------------------------------------------------------------------------------
print("\n" + "="*80)
print("📊 THE EXACT COORDINATES (Averages immediately before the event)")
print("="*80)

wins = master_df[master_df['Target'] == 1]
losses = master_df[master_df['Target'] == -1]

print(f"{'Metric':<15} | {'Right Before a >15% WIN':<25} | {'Right Before a <-15% CRASH':<25}")
print("-" * 75)

for feat in features:
    win_avg = wins[feat].mean()
    loss_avg = losses[feat].mean()
    
    # Format nicely
    if feat == 'Vol_Spike':
        w_str = f"{win_avg:.2f}x Normal"
        l_str = f"{loss_avg:.2f}x Normal"
    elif feat == 'RSI_14':
        w_str = f"{win_avg:.1f}"
        l_str = f"{loss_avg:.1f}"
    else:
        w_str = f"{win_avg:+.2%}"
        l_str = f"{loss_avg:+.2%}"
        
    print(f"{feat:<15} | {w_str:<25} | {l_str:<25}")

print("\n" + "="*80)
print("💡 AI INSIGHTS FOR THE LIVE BOT:")
print("1. Compare the 'Right Before Crash' numbers to the current market.")
print("2. The feature with the highest Importance % is your #1 indicator.")
print("3. We will hardcode these Crash Averages into the final bot as a safety filter.")
print("="*80)

Step 1: Authenticating with Alpaca and Downloading Data...
✓ Downloaded 12412 rows of historical data.

Step 2: Calculating Technical Indicators and Future Outcomes...
Total extreme events found: 3435

Step 3: Training AI Gradient Boosting Model to rank importance...

🧠 AI FEATURE IMPORTANCE (What actually predicts a Win vs Wipeout)
Dist_SMA200     : 24.83%
Vol_1M          : 22.29%
Ret_3M          : 20.30%
Ret_1M          : 11.77%
RSI_14          : 8.62%
Dist_SMA50      : 7.50%
Vol_Spike       : 4.69%

📊 THE EXACT COORDINATES (Averages immediately before the event)
Metric          | Right Before a >15% WIN   | Right Before a <-15% CRASH
---------------------------------------------------------------------------
Ret_1M          | +1.59%                    | +3.76%                   
Ret_3M          | +1.93%                    | +9.14%                   
Vol_1M          | +75.63%                   | +82.22%                  
Dist_SMA50      | -0.83%                    | +2.27%           

In [8]:
# ==============================================================================
# 🚀 CELL 3: THE FINAL PRODUCTION MONTHLY BOT (AI-ENHANCED)
# Run this on the last day of every month to get your trades for the 1st
# ==============================================================================

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
import logging
from kaggle_secrets import UserSecretsClient
import alpaca_trade_api as tradeapi

warnings.filterwarnings('ignore')
logging.getLogger('urllib3').setLevel(logging.WARNING)

# ------------------------------------------------------------------------------
# 1. AUTHENTICATE & PREP UNIVERSE
# ------------------------------------------------------------------------------
print("Step 1: Authenticating with Alpaca...")
try:
    user_secrets = UserSecretsClient()
    API_KEY = user_secrets.get_secret("APCA_API_KEY_ID")
    SECRET_KEY = user_secrets.get_secret("APCA_API_SECRET_KEY")
except Exception as e:
    API_KEY = "YOUR_API_KEY"       
    SECRET_KEY = "YOUR_SECRET_KEY"

api = tradeapi.REST(API_KEY, SECRET_KEY, base_url='https://paper-api.alpaca.markets', api_version='v2')

# Use the Curated Growth & Leverage Universe
universe = [
    'NVDA', 'AAPL', 'MSFT', 'AMZN', 'TSLA', 'META', 'GOOGL', 'AMD', 'AVGO', 'PLTR', 
    'SMCI', 'CRWD', 'SNOW', 'MSTR', 'COIN', 'MARA', 'RIOT',
    'SPY', 'QQQ', 'IWM', 'TLT', 'GLD',
    'TQQQ', 'SOXL', 'UPRO', 'TECL', 'FAS', 'DPST', 'NAIL'
]
# Add VIX proxy if possible (VIXY) since Alpaca free doesn't give ^VIX easily
universe.append('VIXY') 

# ------------------------------------------------------------------------------
# 2. DOWNLOAD RECENT DATA
# ------------------------------------------------------------------------------
print("Step 2: Downloading last 250 days of data for AI indicators...")
end_dt = datetime.today()
start_dt = end_dt - timedelta(days=250) 

prices = pd.DataFrame()

try:
    bars = api.get_bars(universe, tradeapi.TimeFrame.Day, 
                        start_dt.strftime('%Y-%m-%d'), 
                        end_dt.strftime('%Y-%m-%d'), 
                        adjustment='all', feed='iex').df
    
    if not bars.empty:
        bars = bars.reset_index()
        prices = bars.pivot(index='timestamp', columns='symbol', values='close')
        prices.index = pd.to_datetime(prices.index).tz_convert(None).normalize()
        prices.fillna(method='ffill', inplace=True)
except Exception as e:
    print(f"Alpaca API Error: {e}")

if prices.empty:
    print("CRITICAL ERROR: Failed to download market data.")
    exit()

# ------------------------------------------------------------------------------
# 3. CALCULATE SCORES & AI FILTERS
# ------------------------------------------------------------------------------
print("Step 3: Calculating Velocity Scores and checking AI Red Flags...")

# Get latest data (last 64 days for momentum, last 200 for AI filters)
try:
    ret_1m = prices.pct_change(21).iloc[-1]
    ret_3m = prices.pct_change(63).iloc[-1]
    
    # Velocity Score Formula
    scores = (ret_1m * 0.6) + (ret_3m * 0.4)
    scores = scores.dropna()
    
    # AI Indicator: Volatility
    vol_1m = prices.pct_change().rolling(21).std().iloc[-1] * np.sqrt(252)
    
    # AI Indicator: Distance from 200-SMA
    sma_200 = prices.rolling(200).mean().iloc[-1]
    dist_sma200 = (prices.iloc[-1] - sma_200) / sma_200
    
    # Oracle: VIX/VIXY Check (Is the market generally panicked?)
    market_panicking = False
    if 'VIXY' in prices.columns:
        vixy_1m_change = prices['VIXY'].pct_change(21).iloc[-1]
        if vixy_1m_change > 0.30: # If VIXY spiked 30% in a month
            market_panicking = True

except Exception as e:
    print(f"Calculation Error: {e}")
    exit()

# ------------------------------------------------------------------------------
# 4. RANK AND FILTER
# ------------------------------------------------------------------------------
ranked = scores.sort_values(ascending=False)
ranked = ranked.drop('VIXY', errors='ignore')

# Hardcoded AI Limits from Cell 2
MAX_VOL = 0.82    # 82% annualized volatility is the crash zone
MAX_STRETCH = 0.40 # 40% above the 200 SMA means rubber band snap

print("\n" + "="*85)
print(f"🤖 AI-ENHANCED MONTHLY ALLOCATION BOT 🤖")
print(f"Date: {datetime.today().strftime('%Y-%m-%d')}")
print("="*85)

if market_panicking:
    print("🚨 VIX ORACLE ALERT: Extreme Market Panic Detected.")
    print("ACTION: Move 100% of portfolio to CASH / TLT.")
elif ranked.max() <= 0:
    print("🚨 MOMENTUM ALERT: No assets have positive momentum.")
    print("ACTION: Move 100% of portfolio to CASH / TLT.")
else:
    # Check assets from highest score down
    target_asset = None
    
    print("Scanning Top Assets against AI Safety Limits:")
    print(f"{'Ticker':<8} | {'Score':<8} | {'Vol_1M':<8} | {'Dist_SMA200':<12} | {'Status':<15}")
    print("-" * 65)
    
    for ticker in ranked.index[:10]:
        score = ranked[ticker]
        t_vol = vol_1m[ticker]
        t_dist = dist_sma200[ticker]
        
        # Check against AI limits
        if t_vol > MAX_VOL:
            status = "❌ FAIL (Hyper-Vol)"
        elif t_dist > MAX_STRETCH:
            status = "❌ FAIL (Stretched)"
        else:
            status = "✅ PASS"
            if target_asset is None:
                target_asset = ticker # We found our winner!
                
        print(f"{ticker:<8} | {score:+.1%} | {t_vol:+.1%} | {t_dist:+.1%} | {status}")
        
    print("\n" + "="*85)
    if target_asset:
        print(f"💰 THE FINAL AI VERDICT: ALLOCATE 100% TO {target_asset}")
        print("Note: If using Alpaca Margin, ensure Leverage is strictly 1.0x to avoid wipeouts.")
    else:
        print("🚨 ALL TOP 10 ASSETS FAILED AI SAFETY CHECKS.")
        print("The market is a 'Bull Trap'. Move 100% of portfolio to CASH.")
    print("="*85)


Step 1: Authenticating with Alpaca...
Step 2: Downloading last 250 days of data for AI indicators...
Step 3: Calculating Velocity Scores and checking AI Red Flags...

🤖 AI-ENHANCED MONTHLY ALLOCATION BOT 🤖
Date: 2026-02-20
Scanning Top Assets against AI Safety Limits:
Ticker   | Score    | Vol_1M   | Dist_SMA200  | Status         
-----------------------------------------------------------------
SOXL     | +41.6% | +107.5% | +nan% | ❌ FAIL (Hyper-Vol)
DPST     | +32.3% | +62.2% | +nan% | ✅ PASS
NAIL     | +24.0% | +74.2% | +nan% | ✅ PASS
GLD      | +13.4% | +55.4% | +nan% | ✅ PASS
META     | +8.1% | +49.9% | +nan% | ✅ PASS
IWM      | +4.8% | +19.5% | +nan% | ✅ PASS
UPRO     | +4.7% | +35.9% | +nan% | ✅ PASS
NVDA     | +4.0% | +38.5% | +nan% | ✅ PASS
AAPL     | +3.8% | +31.9% | +nan% | ✅ PASS
TLT      | +2.3% | +8.8% | +nan% | ✅ PASS

💰 THE FINAL AI VERDICT: ALLOCATE 100% TO DPST
Note: If using Alpaca Margin, ensure Leverage is strictly 1.0x to avoid wipeouts.


In [9]:
# ==============================================================================
# 🚀 CELL 4: THE ULTIMATE AI BACKTEST
# Compares the Pure Strategy vs the AI-Enhanced Strategy over historical markets
# ==============================================================================

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
import logging
from kaggle_secrets import UserSecretsClient
import alpaca_trade_api as tradeapi

warnings.filterwarnings('ignore')
logging.getLogger('urllib3').setLevel(logging.WARNING)

# ------------------------------------------------------------------------------
# 1. INITIALIZE ALPACA CLIENT
# ------------------------------------------------------------------------------
print("Step 1: Authenticating with Alpaca...")
try:
    user_secrets = UserSecretsClient()
    API_KEY = user_secrets.get_secret("APCA_API_KEY_ID")
    SECRET_KEY = user_secrets.get_secret("APCA_API_SECRET_KEY")
except Exception as e:
    API_KEY = "YOUR_API_KEY"       
    SECRET_KEY = "YOUR_SECRET_KEY"

api = tradeapi.REST(API_KEY, SECRET_KEY, base_url='https://paper-api.alpaca.markets', api_version='v2')

universe = [
    'NVDA', 'AAPL', 'MSFT', 'AMZN', 'TSLA', 'META', 'GOOGL', 'AMD', 'AVGO', 'PLTR', 
    'SMCI', 'CRWD', 'SNOW', 'MSTR', 'COIN', 'MARA', 'RIOT',
    'SPY', 'QQQ', 'IWM', 'TLT', 'GLD',
    'TQQQ', 'SOXL', 'UPRO', 'TECL', 'FAS', 'DPST', 'NAIL'
]

# ------------------------------------------------------------------------------
# 2. BULK DOWNLOAD 10 YEARS OF DATA
# ------------------------------------------------------------------------------
print("Step 2: Downloading 10 Years of Historical Data...")
end_dt = datetime.today() - timedelta(days=1)
start_dt = end_dt - timedelta(days=365 * 10) 

prices = pd.DataFrame()
try:
    bars = api.get_bars(universe, tradeapi.TimeFrame.Day, 
                        start_dt.strftime('%Y-%m-%d'), 
                        end_dt.strftime('%Y-%m-%d'), 
                        adjustment='all', feed='iex').df
    
    if not bars.empty:
        bars = bars.reset_index()
        prices = bars.pivot(index='timestamp', columns='symbol', values='close')
        prices.index = pd.to_datetime(prices.index).tz_convert(None).normalize()
        prices.fillna(method='ffill', inplace=True)
        prices.dropna(axis=1, how='all', inplace=True)
except Exception as e:
    print(f"Alpaca API Error: {e}")

# Pre-calculate indicators for speed
ret_1m_all = prices.pct_change(21)
ret_3m_all = prices.pct_change(63)
scores_all = (ret_1m_all * 0.6) + (ret_3m_all * 0.4)
vol_1m_all = prices.pct_change().rolling(21).std() * np.sqrt(252)
sma_200_all = prices.rolling(200).mean()
dist_sma200_all = (prices - sma_200_all) / sma_200_all

# ------------------------------------------------------------------------------
# 3. SIMULATION ENGINE
# ------------------------------------------------------------------------------
def run_simulation(start_date_str, strategy='PURE', leverage=1.0):
    if prices.empty: return 0
    sim_start = pd.to_datetime(start_date_str)
    if sim_start < prices.index[0]: sim_start = prices.index[0] 
        
    dates = prices.index
    rebalance_dates = pd.Series(index=dates, data=1).resample('BM').last().index
    
    valid_dates = []
    for d in rebalance_dates:
        if d >= sim_start:
            loc = prices.index.get_indexer([d], method='pad')[0]
            if loc != -1: valid_dates.append(prices.index[loc])
                
    valid_dates = sorted(list(set(valid_dates)))
    if len(valid_dates) < 2: return 5000.0 
    
    capital = 5000.0
    portfolio_ticker = None
    
    for i in range(1, len(valid_dates)):
        prev_date = valid_dates[i-1]
        curr_date = valid_dates[i]
        
        # 1. Update Capital
        if portfolio_ticker is not None:
            p0 = prices.loc[prev_date, portfolio_ticker]
            p1 = prices.loc[curr_date, portfolio_ticker]
            if pd.notna(p0) and pd.notna(p1) and p0 > 0:
                trade_return = (p1 / p0) - 1.0
                capital *= (1.0 + (trade_return * leverage))
                if capital <= 0: return 0.0 # Wipeout
                
        # 2. Ranking & Filtering
        loc = prices.index.get_loc(curr_date)
        if loc < 201: # Need 200 days for SMA
            portfolio_ticker = None
            continue
            
        current_scores = scores_all.loc[curr_date]
        current_prices = prices.loc[curr_date]
        current_vol = vol_1m_all.loc[curr_date]
        current_dist = dist_sma200_all.loc[curr_date]
        
        # Base Filter: Must be > $5, must have positive momentum
        valid_assets = current_scores[(current_scores > 0) & (current_prices > 5.0)].index
        
        if strategy == 'AI_ENHANCED':
            # Apply AI Safety Limits (The Crash Predictors)
            safe_assets = []
            for t in valid_assets:
                if current_vol[t] <= 0.82 and current_dist[t] <= 0.40:
                    safe_assets.append(t)
            valid_assets = safe_assets
            
        if len(valid_assets) == 0:
            portfolio_ticker = None # Cash
        else:
            portfolio_ticker = current_scores[valid_assets].idxmax()
            
    return capital

# ------------------------------------------------------------------------------
# 4. RUN EXPERIMENT
# ------------------------------------------------------------------------------
if not prices.empty:
    cohorts = {
        "25 Years Ago*": (datetime.today() - timedelta(days=365*25)).strftime('%Y-%m-%d'),
        "10 Years Ago": (datetime.today() - timedelta(days=365*10)).strftime('%Y-%m-%d'),
        "5 Years Ago": (datetime.today() - timedelta(days=365*5)).strftime('%Y-%m-%d'),
        "1 Year Ago": (datetime.today() - timedelta(days=365)).strftime('%Y-%m-%d'),
        "6 Months Ago": (datetime.today() - timedelta(days=180)).strftime('%Y-%m-%d'),
        "1 Month Ago": (datetime.today() - timedelta(days=30)).strftime('%Y-%m-%d')
    }

    print("\n" + "="*105)
    print(f"🏆 THE FINAL SHOWDOWN: PURE MOMENTUM vs AI-ENHANCED (Starting Capital: $5,000)")
    print("="*105)
    print(f"{'Timeframe':<15} | {'Pure (1.0x Cash)':<20} | {'AI-Enhanced (1.0x Cash)':<22} | {'AI-Enhanced (2.0x Margin)':<22}")
    print("-" * 105)

    for label, d_str in cohorts.items():
        res_pure_1x = run_simulation(d_str, 'PURE', 1.0)
        res_ai_1x = run_simulation(d_str, 'AI_ENHANCED', 1.0)
        res_ai_2x = run_simulation(d_str, 'AI_ENHANCED', 2.0)
        
        str_p1 = f"${res_pure_1x:,.2f}" if res_pure_1x > 0 else "WIPED OUT ($0)"
        str_a1 = f"${res_ai_1x:,.2f}" if res_ai_1x > 0 else "WIPED OUT ($0)"
        str_a2 = f"${res_ai_2x:,.2f}" if res_ai_2x > 0 else "WIPED OUT ($0)"
        
        print(f"{label:<15} | {str_p1:<20} | {str_a1:<22} | {str_a2:<22}")
            
    print("-" * 105)
    print("\n* Note: Alpaca Free IEX data only goes back to ~2016.")
    print("EXPECTED OUTCOME:")
    print("1. AI-Enhanced (1.0x) should have slightly lower peak returns than Pure (1.0x) during massive irrational bull runs, but vastly smaller drawdowns.")
    print("2. The ultimate question is the 3rd column: Does the AI filter save the 2.0x Margin account from wiping out during the 2022 crash?")


Step 1: Authenticating with Alpaca...
Step 2: Downloading 10 Years of Historical Data...

🏆 THE FINAL SHOWDOWN: PURE MOMENTUM vs AI-ENHANCED (Starting Capital: $5,000)
Timeframe       | Pure (1.0x Cash)     | AI-Enhanced (1.0x Cash) | AI-Enhanced (2.0x Margin)
---------------------------------------------------------------------------------------------------------
25 Years Ago*   | $4,377.28            | $11,234.99             | $8,667.16             
10 Years Ago    | $4,377.28            | $11,234.99             | $8,667.16             
5 Years Ago     | $4,377.28            | $11,234.99             | $8,667.16             
1 Year Ago      | $5,420.60            | $7,188.86              | $9,761.81             
6 Months Ago    | $5,025.12            | $5,869.30              | $6,760.96             
1 Month Ago     | $5,000.00            | $5,000.00              | $5,000.00             
---------------------------------------------------------------------------------------------------

In [10]:
# ==============================================================================
# 🚀 CELL 5: AI-ENHANCED FREQUENCY SHOWDOWN
# Does "Monthly" still win when the AI prevents margin wipeouts?
# ==============================================================================

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
import logging
from kaggle_secrets import UserSecretsClient
import alpaca_trade_api as tradeapi

warnings.filterwarnings('ignore')
logging.getLogger('urllib3').setLevel(logging.WARNING)

# ------------------------------------------------------------------------------
# 1. INITIALIZE ALPACA CLIENT & UNIVERSE
# ------------------------------------------------------------------------------
print("Step 1: Authenticating with Alpaca...")
try:
    user_secrets = UserSecretsClient()
    API_KEY = user_secrets.get_secret("APCA_API_KEY_ID")
    SECRET_KEY = user_secrets.get_secret("APCA_API_SECRET_KEY")
except Exception as e:
    API_KEY = "YOUR_API_KEY"       
    SECRET_KEY = "YOUR_SECRET_KEY"

api = tradeapi.REST(API_KEY, SECRET_KEY, base_url='https://paper-api.alpaca.markets', api_version='v2')

universe = [
    'NVDA', 'AAPL', 'MSFT', 'AMZN', 'TSLA', 'META', 'GOOGL', 'AMD', 'AVGO', 'PLTR', 
    'SMCI', 'CRWD', 'SNOW', 'MSTR', 'COIN', 'MARA', 'RIOT',
    'SPY', 'QQQ', 'IWM', 'TLT', 'GLD',
    'TQQQ', 'SOXL', 'UPRO', 'TECL', 'FAS', 'DPST', 'NAIL'
]

# ------------------------------------------------------------------------------
# 2. BULK DOWNLOAD 10 YEARS OF DATA
# ------------------------------------------------------------------------------
print("Step 2: Downloading 10 Years of Historical Data...")
end_dt = datetime.today() - timedelta(days=1)
start_dt = end_dt - timedelta(days=365 * 10) 

prices = pd.DataFrame()
try:
    bars = api.get_bars(universe, tradeapi.TimeFrame.Day, 
                        start_dt.strftime('%Y-%m-%d'), 
                        end_dt.strftime('%Y-%m-%d'), 
                        adjustment='all', feed='iex').df
    
    if not bars.empty:
        bars = bars.reset_index()
        prices = bars.pivot(index='timestamp', columns='symbol', values='close')
        prices.index = pd.to_datetime(prices.index).tz_convert(None).normalize()
        prices.fillna(method='ffill', inplace=True)
        prices.dropna(axis=1, how='all', inplace=True)
except Exception as e:
    print(f"Alpaca API Error: {e}")

# Pre-calculate AI indicators for speed
ret_1m_all = prices.pct_change(21)
ret_3m_all = prices.pct_change(63)
scores_all = (ret_1m_all * 0.6) + (ret_3m_all * 0.4)
vol_1m_all = prices.pct_change().rolling(21).std() * np.sqrt(252)
sma_200_all = prices.rolling(200).mean()
dist_sma200_all = (prices - sma_200_all) / sma_200_all

# ------------------------------------------------------------------------------
# 3. SIMULATION ENGINE (2.0x MARGIN + AI FILTERS)
# ------------------------------------------------------------------------------
def run_ai_frequency_sim(start_date_str, freq='M'):
    if prices.empty: return 0
    sim_start = pd.to_datetime(start_date_str)
    if sim_start < prices.index[0]: sim_start = prices.index[0] 
        
    dates = prices.index
    if freq == 'D': rule = 'B'
    elif freq == 'W': rule = 'W-FRI'
    elif freq == 'M': rule = 'BM'
    elif freq == 'Y': rule = 'BY'
    
    rebalance_dates = pd.Series(index=dates, data=1).resample(rule).last().index
    
    valid_dates = []
    for d in rebalance_dates:
        if d >= sim_start:
            loc = prices.index.get_indexer([d], method='pad')[0]
            if loc != -1: valid_dates.append(prices.index[loc])
                
    valid_dates = sorted(list(set(valid_dates)))
    if len(valid_dates) < 2: return 5000.0 
    
    capital = 5000.0
    portfolio_ticker = None
    
    for i in range(1, len(valid_dates)):
        prev_date = valid_dates[i-1]
        curr_date = valid_dates[i]
        
        # 1. Update Capital (ALWAYS 2.0x MARGIN FOR THIS TEST)
        if portfolio_ticker is not None:
            p0 = prices.loc[prev_date, portfolio_ticker]
            p1 = prices.loc[curr_date, portfolio_ticker]
            if pd.notna(p0) and pd.notna(p1) and p0 > 0:
                trade_return = (p1 / p0) - 1.0
                capital *= (1.0 + (trade_return * 2.0)) # 2.0x Leverage
                if capital <= 0: return 0.0 
                
        # 2. Ranking & AI Filtering
        loc = prices.index.get_loc(curr_date)
        if loc < 201: # Need 200 days for SMA
            portfolio_ticker = None
            continue
            
        current_scores = scores_all.loc[curr_date]
        current_prices = prices.loc[curr_date]
        current_vol = vol_1m_all.loc[curr_date]
        current_dist = dist_sma200_all.loc[curr_date]
        
        # Base Filter & AI Filters
        valid_assets = current_scores[(current_scores > 0) & (current_prices > 5.0)].index
        safe_assets = []
        for t in valid_assets:
            if current_vol[t] <= 0.82 and current_dist[t] <= 0.40:
                safe_assets.append(t)
        
        if len(safe_assets) == 0:
            portfolio_ticker = None # Cash
        else:
            portfolio_ticker = current_scores[safe_assets].idxmax()
            
    return capital

# ------------------------------------------------------------------------------
# 4. RUN EXPERIMENT
# ------------------------------------------------------------------------------
if not prices.empty:
    cohorts = {
        "10 Years Ago": (datetime.today() - timedelta(days=365*10)).strftime('%Y-%m-%d'),
        "5 Years Ago": (datetime.today() - timedelta(days=365*5)).strftime('%Y-%m-%d'),
        "1 Year Ago": (datetime.today() - timedelta(days=365)).strftime('%Y-%m-%d'),
        "6 Months Ago": (datetime.today() - timedelta(days=180)).strftime('%Y-%m-%d')
    }

    print("\n" + "="*95)
    print(f"🤖 THE FINAL AI FREQUENCY SHOWDOWN (2.0x Margin Leverage Active)")
    print("="*95)
    print(f"{'Timeframe':<15} | {'Daily Check':<16} | {'Weekly Check':<16} | {'Monthly Check':<16} | {'Yearly Check':<16}")
    print("-" * 95)

    for label, d_str in cohorts.items():
        res_d = run_ai_frequency_sim(d_str, 'D')
        res_w = run_ai_frequency_sim(d_str, 'W')
        res_m = run_ai_frequency_sim(d_str, 'M')
        res_y = run_ai_frequency_sim(d_str, 'Y')
        
        str_d = f"${res_d:,.2f}" if res_d > 0 else "WIPED OUT ($0)"
        str_w = f"${res_w:,.2f}" if res_w > 0 else "WIPED OUT ($0)"
        str_m = f"${res_m:,.2f}" if res_m > 0 else "WIPED OUT ($0)"
        str_y = f"${res_y:,.2f}" if res_y > 0 else "WIPED OUT ($0)"
        
        # Highlight max
        vals = [res_d, res_w, res_m, res_y]
        max_val = max(vals)
        
        if res_d == max_val and res_d > 0: str_d = f"🏆 {str_d}"
        if res_w == max_val and res_w > 0: str_w = f"🏆 {str_w}"
        if res_m == max_val and res_m > 0: str_m = f"🏆 {str_m}"
        if res_y == max_val and res_y > 0: str_y = f"🏆 {str_y}"
        
        print(f"{label:<15} | {str_d:<16} | {str_w:<16} | {str_m:<16} | {str_y:<16}")
            
    print("-" * 95)


Step 1: Authenticating with Alpaca...
Step 2: Downloading 10 Years of Historical Data...

🤖 THE FINAL AI FREQUENCY SHOWDOWN (2.0x Margin Leverage Active)
Timeframe       | Daily Check      | Weekly Check     | Monthly Check    | Yearly Check    
-----------------------------------------------------------------------------------------------
10 Years Ago    | $572.89          | $2,383.09        | $8,667.16        | 🏆 $40,594.76    
5 Years Ago     | $572.89          | $2,383.09        | $8,667.16        | 🏆 $55,261.95    
1 Year Ago      | $985.47          | $4,285.46        | 🏆 $9,761.81      | $5,000.00       
6 Months Ago    | $2,460.21        | $4,477.93        | 🏆 $6,760.96      | $5,000.00       
-----------------------------------------------------------------------------------------------


In [11]:
# ==============================================================================
# 🚀 CELL 6: INSTITUTIONAL ALPHA ENGINE
# Upgrading the bot to hunt for Relative Strength, Volume Accumulation, and VCP
# ==============================================================================

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
from kaggle_secrets import UserSecretsClient
import alpaca_trade_api as tradeapi

warnings.filterwarnings('ignore')

print("Step 1: Authenticating and Fetching Alpha Data...")
try:
    user_secrets = UserSecretsClient()
    API_KEY = user_secrets.get_secret("APCA_API_KEY_ID")
    SECRET_KEY = user_secrets.get_secret("APCA_API_SECRET_KEY")
except:
    API_KEY = "YOUR_API_KEY"       
    SECRET_KEY = "YOUR_SECRET_KEY"

api = tradeapi.REST(API_KEY, SECRET_KEY, base_url='https://paper-api.alpaca.markets', api_version='v2')

# Universe + Market Proxy (SPY)
universe = [
    'NVDA', 'AAPL', 'MSFT', 'AMZN', 'TSLA', 'META', 'GOOGL', 'AMD', 'AVGO', 'PLTR', 
    'SMCI', 'CRWD', 'SNOW', 'MSTR', 'COIN', 'MARA', 'RIOT',
    'TQQQ', 'SOXL', 'UPRO', 'TECL', 'FAS', 'DPST', 'NAIL', 'SPY'
]

end_dt = datetime.today()
start_dt = end_dt - timedelta(days=250) 

prices = pd.DataFrame()
volumes = pd.DataFrame()

try:
    bars = api.get_bars(universe, tradeapi.TimeFrame.Day, 
                        start_dt.strftime('%Y-%m-%d'), 
                        end_dt.strftime('%Y-%m-%d'), 
                        adjustment='all', feed='iex').df
    
    if not bars.empty:
        bars = bars.reset_index()
        # Extract both Price and Volume
        prices = bars.pivot(index='timestamp', columns='symbol', values='close')
        volumes = bars.pivot(index='timestamp', columns='symbol', values='volume')
        
        prices.index = pd.to_datetime(prices.index).tz_convert(None).normalize()
        volumes.index = pd.to_datetime(volumes.index).tz_convert(None).normalize()
        
        prices.fillna(method='ffill', inplace=True)
        volumes.fillna(0, inplace=True)
except Exception as e:
    print(f"Alpaca API Error: {e}")

# ------------------------------------------------------------------------------
# 2. CALCULATE INSTITUTIONAL ALPHA FACTORS
# ------------------------------------------------------------------------------
print("Step 2: Calculating Advanced Hedge Fund Metrics...")

# 1. Base Momentum
ret_1m = prices.pct_change(21).iloc[-1]
ret_3m = prices.pct_change(63).iloc[-1]

# 2. Relative Strength vs SPY (The Ultimate Tell)
spy_ret_1m = prices['SPY'].pct_change(21).iloc[-1]
spy_ret_3m = prices['SPY'].pct_change(63).iloc[-1]

rs_1m = ret_1m - spy_ret_1m
rs_3m = ret_3m - spy_ret_3m

# 3. Volume Accumulation/Distribution (Whale Footprints)
# Sum of volume on Up days vs Sum of volume on Down days over last 21 days
recent_prices = prices.iloc[-22:]
recent_vols = volumes.iloc[-22:]

price_diff = recent_prices.diff()
up_volume = recent_vols[price_diff > 0].sum()
down_volume = recent_vols[price_diff < 0].sum()

# Avoid div by zero
down_volume = down_volume.replace(0, 1)
vol_accumulation_ratio = up_volume / down_volume

# 4. Volatility Contraction (The Spring Coil)
# If 1-month vol is significantly lower than 3-month vol, the stock is tightening
vol_1m = prices.pct_change().rolling(21).std().iloc[-1]
vol_3m = prices.pct_change().rolling(63).std().iloc[-1]
vcp_squeeze = vol_3m / vol_1m.replace(0, np.nan) # Higher number = tighter squeeze right now

# 5. The Rubber Band Safety Check (Distance from 200 SMA)
sma_200 = prices.rolling(200).mean().iloc[-1]
dist_sma200 = (prices.iloc[-1] - sma_200) / sma_200

# ------------------------------------------------------------------------------
# 3. GENERATE THE CONVEXITY SCORE
# ------------------------------------------------------------------------------
print("Step 3: Generating Final Convexity Rankings...\n")

# Combine into a scoring dataframe
scoring_df = pd.DataFrame({
    'Base_Mom': (ret_1m * 0.6) + (ret_3m * 0.4),
    'Rel_Strength': (rs_1m * 0.6) + (rs_3m * 0.4),
    'Whale_Volume': vol_accumulation_ratio,
    'VCP_Squeeze': vcp_squeeze,
    'Danger_Stretch': dist_sma200,
    'Vol_1M': vol_1m * np.sqrt(252) # Annualized for the safety check
})

# Drop SPY as a target (it's just the benchmark)
if 'SPY' in scoring_df.index: scoring_df = scoring_df.drop('SPY')
scoring_df.dropna(inplace=True)

# Normalize the scores so we can combine them into one Master "Alpha Score"
# Using simple Min-Max scaling for blending
def normalize(series):
    return (series - series.min()) / (series.max() - series.min())

scoring_df['Alpha_Score'] = (
    (normalize(scoring_df['Rel_Strength']) * 0.40) +  # 40% weight to outperforming the market
    (normalize(scoring_df['Whale_Volume']) * 0.40) +  # 40% weight to massive institutional buying
    (normalize(scoring_df['VCP_Squeeze']) * 0.20)     # 20% weight to volatility squeezing
)

# Apply AI Safety Filters (from previous step)
MAX_VOL = 0.82
MAX_STRETCH = 0.40

# Filter out the dangerous ones
safe_df = scoring_df[(scoring_df['Vol_1M'] <= MAX_VOL) & (scoring_df['Danger_Stretch'] <= MAX_STRETCH)]
ranked_alpha = safe_df.sort_values('Alpha_Score', ascending=False)

print("="*105)
print("🐋 INSTITUTIONAL ALPHA RANKINGS (Hunting the Next Multi-Bagger) 🐋")
print("="*105)
print(f"{'Ticker':<8} | {'Alpha Score':<12} | {'Rel. Strength':<15} | {'Whale Volume':<15} | {'VCP Squeeze':<15}")
print("-" * 105)

for ticker in ranked_alpha.index[:10]:
    a_score = ranked_alpha.loc[ticker, 'Alpha_Score'] * 100
    rs = ranked_alpha.loc[ticker, 'Rel_Strength']
    wv = ranked_alpha.loc[ticker, 'Whale_Volume']
    vcp = ranked_alpha.loc[ticker, 'VCP_Squeeze']
    
    # Formatting
    rs_str = f"Beating SPY by {rs:+.1%}" if rs > 0 else f"Losing to SPY {rs:+.1%}"
    wv_str = f"{wv:.1f}x Buyers vs Sellers"
    vcp_str = f"Coiling" if vcp > 1.2 else "Normal"
    
    print(f"{ticker:<8} | {a_score:>6.1f} / 100 | {rs_str:<15} | {wv_str:<15} | {vcp_str:<15}")

print("="*105)
if not ranked_alpha.empty:
    top_pick = ranked_alpha.index[0]
    print(f"🎯 THE SMART MONEY VERDICT: ALLOCATE TO {top_pick}")
    print(f"Why? It is surviving the safety filters, beating the S&P 500, and has the strongest institutional buying volume.")
else:
    print("🚨 NO ASSETS PASSED THE ALPHA + SAFETY FILTERS. GO TO CASH.")


Step 1: Authenticating and Fetching Alpha Data...
Step 2: Calculating Advanced Hedge Fund Metrics...
Step 3: Generating Final Convexity Rankings...

🐋 INSTITUTIONAL ALPHA RANKINGS (Hunting the Next Multi-Bagger) 🐋
Ticker   | Alpha Score  | Rel. Strength   | Whale Volume    | VCP Squeeze    
---------------------------------------------------------------------------------------------------------
🚨 NO ASSETS PASSED THE ALPHA + SAFETY FILTERS. GO TO CASH.


In [12]:
# ==============================================================================
# 🚀 CELL 7: THE TITAN UNIFIED DISCOVERY ENGINE (LIVE ML PREDICTION)
# Uses Gradient Boosting to learn causal drivers of 15% monthly jumps
# ==============================================================================

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from sklearn.ensemble import GradientBoostingClassifier
import warnings
import logging
from kaggle_secrets import UserSecretsClient
import alpaca_trade_api as tradeapi

warnings.filterwarnings('ignore')
logging.getLogger('urllib3').setLevel(logging.WARNING)

# ------------------------------------------------------------------------------
# 1. AUTHENTICATE & DOWNLOAD DATA
# ------------------------------------------------------------------------------
print("Step 1: Authenticating with Alpaca...")
try:
    user_secrets = UserSecretsClient()
    API_KEY = user_secrets.get_secret("APCA_API_KEY_ID")
    SECRET_KEY = user_secrets.get_secret("APCA_API_SECRET_KEY")
except Exception as e:
    API_KEY = "YOUR_API_KEY"       
    SECRET_KEY = "YOUR_SECRET_KEY"

api = tradeapi.REST(API_KEY, SECRET_KEY, base_url='https://paper-api.alpaca.markets', api_version='v2')

# Curated Universe + Market Proxy
universe = [
    'NVDA', 'AAPL', 'MSFT', 'AMZN', 'TSLA', 'META', 'GOOGL', 'AMD', 'AVGO', 'PLTR', 
    'SMCI', 'CRWD', 'SNOW', 'MSTR', 'COIN', 'MARA', 'RIOT',
    'TQQQ', 'SOXL', 'UPRO', 'TECL', 'FAS', 'DPST', 'NAIL', 'SPY'
]

print("Step 2: Downloading 10 Years of Data to Train the Titan Engine...")
end_dt = datetime.today()
start_dt = end_dt - timedelta(days=365 * 10) 

prices = pd.DataFrame()
volumes = pd.DataFrame()

try:
    bars = api.get_bars(universe, tradeapi.TimeFrame.Day, 
                        start_dt.strftime('%Y-%m-%d'), 
                        end_dt.strftime('%Y-%m-%d'), 
                        adjustment='all', feed='iex').df
    
    if not bars.empty:
        bars = bars.reset_index()
        prices = bars.pivot(index='timestamp', columns='symbol', values='close')
        volumes = bars.pivot(index='timestamp', columns='symbol', values='volume')
        
        prices.index = pd.to_datetime(prices.index).tz_convert(None).normalize()
        volumes.index = pd.to_datetime(volumes.index).tz_convert(None).normalize()
        
        prices.fillna(method='ffill', inplace=True)
        volumes.fillna(0, inplace=True)
except Exception as e:
    print(f"Alpaca API Error: {e}")

# ------------------------------------------------------------------------------
# 2. FEATURE ENGINEERING (ATTRIBUTION FRAMEWORK)
# ------------------------------------------------------------------------------
print("Step 3: Calculating Causal Features & Structural Anomalies...")

all_training_data = []

# Pre-calculate SPY for relative strength
spy_ret_1m = prices['SPY'].pct_change(21)

for ticker in universe:
    if ticker == 'SPY': continue
        
    df = pd.DataFrame({'Close': prices[ticker], 'Volume': volumes[ticker]})
    if len(df) < 252: continue
        
    # Feature 1: Raw Momentum
    df['Mom_1M'] = df['Close'].pct_change(21)
    df['Mom_3M'] = df['Close'].pct_change(63)
    
    # Feature 2: Relative Strength vs Market
    df['Rel_Strength'] = df['Mom_1M'] - spy_ret_1m
    
    # Feature 3: Volatility & VCP Squeeze
    df['Vol_1M'] = df['Close'].pct_change().rolling(21).std() * np.sqrt(252)
    vol_3m = df['Close'].pct_change().rolling(63).std() * np.sqrt(252)
    df['VCP_Squeeze'] = vol_3m / df['Vol_1M'].replace(0, np.nan)
    
    # Feature 4: Volume Accumulation (Whale Buying)
    price_diff = df['Close'].diff()
    up_vol = df['Volume'].where(price_diff > 0, 0).rolling(21).sum()
    down_vol = df['Volume'].where(price_diff < 0, 0).rolling(21).sum()
    df['Whale_Ratio'] = up_vol / down_vol.replace(0, 1)
    
    # Feature 5: The Rubber Band (Distance from 200 SMA)
    sma_200 = df['Close'].rolling(200).mean()
    df['Dist_SMA200'] = (df['Close'] - sma_200) / sma_200
    
    # TARGET: Did the stock jump more than 15% next month?
    df['Target_Return'] = df['Close'].shift(-21) / df['Close'] - 1.0
    df['Will_Explode'] = (df['Target_Return'] > 0.15).astype(int)
    
    # Drop NaNs and add to training set
    df.dropna(inplace=True)
    all_training_data.append(df)

master_df = pd.concat(all_training_data)

# ------------------------------------------------------------------------------
# 3. MACHINE LEARNING OPTIMIZATION (TRAINING)
# ------------------------------------------------------------------------------
print("Step 4: Training Gradient Boosting Model on Historical Data...")

# Features used by the ML Model
features = ['Mom_1M', 'Mom_3M', 'Rel_Strength', 'Vol_1M', 'VCP_Squeeze', 'Whale_Ratio', 'Dist_SMA200']

# We train on all data EXCEPT the last 30 days (to prevent data leakage for our live prediction)
train_df = master_df.iloc[:-30]
live_df = master_df.groupby(level=0).last() # Gets the most recent row for each ticker

X_train = train_df[features]
y_train = train_df['Will_Explode']

# Train the Titan Boosting Model
titan_model = GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, max_depth=4, random_state=42)
titan_model.fit(X_train, y_train)

print(f"✓ Model Trained on {len(X_train)} historical trading scenarios.")

# ------------------------------------------------------------------------------
# 4. LIVE PREDICTION (DISCOVERY & INTERVENTION)
# ------------------------------------------------------------------------------
print("\n" + "="*95)
print("🎯 TITAN ML PREDICTION: HUNTING TODAY'S MULTI-BAGGERS 🎯")
print("="*95)

# Get the absolute most recent data point for each ticker
current_predictions = []

for ticker in universe:
    if ticker == 'SPY': continue
    try:
        # Get the last row of features for this ticker
        ticker_data = all_training_data[universe.index(ticker) if ticker in universe else 0].iloc[[-1]]
        X_live = ticker_data[features]
        
        # Predict Probability of >15% Jump
        prob = titan_model.predict_proba(X_live)[0][1]
        
        # Save results
        current_predictions.append({
            'Ticker': ticker,
            'Explosion_Probability': prob * 100,
            'Mom_1M': X_live['Mom_1M'].values[0] * 100,
            'Whale_Ratio': X_live['Whale_Ratio'].values[0],
            'Dist_SMA200': X_live['Dist_SMA200'].values[0] * 100
        })
    except:
        pass

# Rank by Probability
pred_df = pd.DataFrame(current_predictions).sort_values('Explosion_Probability', ascending=False)

print(f"{'Ticker':<8} | {'Probability of >15% Jump':<25} | {'1M Momentum':<15} | {'Whale Volume':<15} | {'Distance from 200SMA'}")
print("-" * 95)

for _, row in pred_df.head(10).iterrows():
    ticker = row['Ticker']
    prob = row['Explosion_Probability']
    mom = row['Mom_1M']
    whale = row['Whale_Ratio']
    dist = row['Dist_SMA200']
    
    # Visual flag for high conviction
    prob_str = f"🔥 {prob:.1f}%" if prob > 30.0 else f"{prob:.1f}%"
    
    print(f"{ticker:<8} | {prob_str:<25} | {mom:+.1f}%{'':<9} | {whale:.1f}x Buyers{'':<3} | {dist:+.1f}%")

print("="*95)

# Final Verdict Logic
top_pick = pred_df.iloc[0]
if top_pick['Explosion_Probability'] > 25.0: # 25% is an incredibly high probability for a massive outlier event
    print(f"💰 ML VERDICT: ALLOCATE 100% TO {top_pick['Ticker']}")
    print(f"The Titan Model has detected a structural alignment. {top_pick['Ticker']} has the exact")
    print(f"features today that historically led to a >15% monthly explosion.")
else:
    print(f"🚨 ML VERDICT: GO TO CASH (TLT / SPY).")
    print(f"The model's highest confidence is only {top_pick['Explosion_Probability']:.1f}%.")
    print(f"The structural drivers required for a breakout are not present in the current market.")
print("="*95)


Step 1: Authenticating with Alpaca...
Step 2: Downloading 10 Years of Data to Train the Titan Engine...
Step 3: Calculating Causal Features & Structural Anomalies...
Step 4: Training Gradient Boosting Model on Historical Data...
✓ Model Trained on 28028 historical trading scenarios.

🎯 TITAN ML PREDICTION: HUNTING TODAY'S MULTI-BAGGERS 🎯
Ticker   | Probability of >15% Jump  | 1M Momentum     | Whale Volume    | Distance from 200SMA
-----------------------------------------------------------------------------------------------
MSTR     | 🔥 36.3%                   | +3.6%          | 2.4x Buyers    | -47.4%
MARA     | 🔥 32.6%                   | +8.5%          | 1.1x Buyers    | -30.0%
RIOT     | 🔥 30.9%                   | +29.2%          | 1.5x Buyers    | +28.1%
AVGO     | 🔥 30.6%                   | -0.1%          | 1.3x Buyers    | +10.0%
PLTR     | 29.2%                     | -10.9%          | 0.6x Buyers    | +5.8%
NAIL     | 28.8%                     | +22.7%          | 1.3x Buyer

In [4]:
# ==============================================================================
# 🚀 CELL 8: THE OUT-OF-SAMPLE TITAN ENGINE (STRICT VALIDATION)
# Fixes Schedule Gaps: Forces immediate day-1 execution and true holding periods.
# Unlocks 10-Year History: Shows In-Sample vs Out-of-Sample reality.
# ==============================================================================

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from sklearn.ensemble import GradientBoostingClassifier
import warnings
import logging
from kaggle_secrets import UserSecretsClient
import alpaca_trade_api as tradeapi

warnings.filterwarnings('ignore')
logging.getLogger('urllib3').setLevel(logging.WARNING)

# ------------------------------------------------------------------------------
# 1. AUTHENTICATE & DOWNLOAD DATA
# ------------------------------------------------------------------------------
print("Step 1: Authenticating with Alpaca & Downloading Data...")
try:
    user_secrets = UserSecretsClient()
    API_KEY = user_secrets.get_secret("APCA_API_KEY_ID")
    SECRET_KEY = user_secrets.get_secret("APCA_API_SECRET_KEY")
except Exception as e:
    API_KEY = "YOUR_API_KEY"       
    SECRET_KEY = "YOUR_SECRET_KEY"

api = tradeapi.REST(API_KEY, SECRET_KEY, base_url='https://paper-api.alpaca.markets', api_version='v2')

universe = [
    'NVDA', 'AAPL', 'MSFT', 'AMZN', 'TSLA', 'META', 'GOOGL', 'AMD', 'AVGO', 'PLTR', 
    'SMCI', 'CRWD', 'SNOW', 'MSTR', 'COIN', 'MARA', 'RIOT',
    'TQQQ', 'SOXL', 'UPRO', 'TECL', 'FAS', 'DPST', 'NAIL', 'SPY'
]

end_dt = datetime.today() - timedelta(days=1)
start_dt = end_dt - timedelta(days=365 * 10) 

prices = pd.DataFrame()
volumes = pd.DataFrame()

try:
    bars = api.get_bars(universe, tradeapi.TimeFrame.Day, 
                        start_dt.strftime('%Y-%m-%d'), 
                        end_dt.strftime('%Y-%m-%d'), 
                        adjustment='all', feed='iex').df
    
    if not bars.empty:
        bars = bars.reset_index()
        prices = bars.pivot(index='timestamp', columns='symbol', values='close')
        volumes = bars.pivot(index='timestamp', columns='symbol', values='volume')
        
        prices.index = pd.to_datetime(prices.index).tz_convert(None).normalize()
        volumes.index = pd.to_datetime(volumes.index).tz_convert(None).normalize()
        
        prices.fillna(method='ffill', inplace=True)
        volumes.fillna(0, inplace=True)
except Exception as e:
    print(f"Alpaca API Error: {e}")

# ------------------------------------------------------------------------------
# 2. FEATURE ENGINEERING (CAUSAL INDICATORS)
# ------------------------------------------------------------------------------
print("Step 2: Engineering Causal Features for the entire timeline...")

all_data = []

if 'SPY' in prices.columns:
    spy_ret_1m = prices['SPY'].pct_change(21)
else:
    spy_ret_1m = pd.Series(0, index=prices.index) 

for ticker in universe:
    if ticker == 'SPY' or ticker not in prices.columns: 
        continue
    
    df = pd.DataFrame({'Close': prices[ticker], 'Volume': volumes[ticker]}, index=prices.index)
    df.dropna(subset=['Close'], inplace=True)
    if len(df) < 252: continue
        
    df['Ticker'] = ticker
    df['Mom_1M'] = df['Close'].pct_change(21)
    df['Mom_3M'] = df['Close'].pct_change(63)
    
    aligned_spy = spy_ret_1m.reindex(df.index)
    df['Rel_Strength'] = df['Mom_1M'] - aligned_spy
    
    df['Vol_1M'] = df['Close'].pct_change().rolling(21).std() * np.sqrt(252)
    vol_3m = df['Close'].pct_change().rolling(63).std() * np.sqrt(252)
    df['VCP_Squeeze'] = vol_3m / df['Vol_1M'].replace(0, np.nan)
    
    price_diff = df['Close'].diff()
    up_vol = df['Volume'].where(price_diff > 0, 0).rolling(21).sum()
    down_vol = df['Volume'].where(price_diff < 0, 0).rolling(21).sum()
    df['Whale_Ratio'] = up_vol / down_vol.replace(0, 1)
    
    sma_200 = df['Close'].rolling(200).mean()
    df['Dist_SMA200'] = (df['Close'] - sma_200) / sma_200
    
    df['Target_Return'] = df['Close'].shift(-21) / df['Close'] - 1.0
    df['Will_Explode'] = (df['Target_Return'] > 0.15).astype(int)
    
    df.dropna(inplace=True)
    all_data.append(df)

master_df = pd.concat(all_data)
master_df.sort_index(inplace=True)

# ------------------------------------------------------------------------------
# 3. TRAINING MODEL & SCORING ENTIRE TIMELINE
# ------------------------------------------------------------------------------
print("Step 3: Training Model and Scoring the entire historical timeline...")
features = ['Mom_1M', 'Mom_3M', 'Rel_Strength', 'Vol_1M', 'VCP_Squeeze', 'Whale_Ratio', 'Dist_SMA200']

min_dt = master_df.index.min()
max_dt = master_df.index.max()
span = max_dt - min_dt

split_date = min_dt + pd.Timedelta(days=int(span.days * 0.60))
train_df = master_df[master_df.index < split_date]

X_train = train_df[features]
y_train = train_df['Will_Explode']

titan_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=42)
titan_model.fit(X_train, y_train)

master_df['ML_Probability'] = titan_model.predict_proba(master_df[features])[:, 1]
ml_scores = master_df.pivot_table(index=master_df.index, columns='Ticker', values='ML_Probability')
ml_scores.fillna(0, inplace=True)

# ------------------------------------------------------------------------------
# 4. SIMULATION ENGINE (FOOLPROOF LOGIC)
# ------------------------------------------------------------------------------
print("Step 4: Running the Backtest across Frequencies...")

def run_ml_sim(start_date_str, freq='M'):
    if ml_scores.empty: return 0
    sim_start = pd.to_datetime(start_date_str)
        
    dates = prices.index
    if freq == 'D': rule = 'B'
    elif freq == 'W': rule = 'W-FRI'
    elif freq == 'M': rule = 'BM'
    elif freq == 'Y': rule = 'BYE' # Updated to modern Pandas standard
    
    ideal_dates = pd.Series(index=dates, data=1).resample(rule).last().index.tolist()
    valid_dates = []
    
    # 1. Force the simulation to check the market on DAY 1
    start_loc = dates.get_indexer([sim_start], method='pad')[0]
    if start_loc != -1: valid_dates.append(dates[start_loc])
        
    # 2. Add all scheduled check-ups
    for d in ideal_dates:
        if d > sim_start:
            loc = prices.index.get_indexer([d], method='pad')[0]
            if loc != -1: valid_dates.append(prices.index[loc])
            
    # 3. Force final Mark-to-Market day
    final_day = prices.index[-1]
    if final_day > sim_start and final_day not in valid_dates:
        valid_dates.append(final_day)
                
    valid_dates = sorted(list(set(valid_dates)))
    if len(valid_dates) < 2: return 5000.0 
    
    capital = 5000.0
    portfolio_ticker = None
    
    # THE FLAWLESS EXECUTION LOOP
    for i in range(len(valid_dates) - 1):
        buy_date = valid_dates[i]
        sell_date = valid_dates[i+1]
        
        # Action A: Analyze Market on Buy Date
        if buy_date in ml_scores.index:
            current_probs = ml_scores.loc[buy_date]
            if current_probs.max() < 0.15: 
                portfolio_ticker = None # Retreat to cash
            else:
                portfolio_ticker = current_probs.idxmax()
        else:
            portfolio_ticker = None
            
        # Action B: Fast forward to Sell Date & Update Capital (2.0x Margin)
        if portfolio_ticker is not None:
            p0 = prices.loc[buy_date, portfolio_ticker]
            p1 = prices.loc[sell_date, portfolio_ticker]
            
            if pd.notna(p0) and pd.notna(p1) and p0 > 0:
                trade_return = (p1 / p0) - 1.0
                capital *= (1.0 + (trade_return * 2.0))
                if capital <= 0: return 0.0 # Margin Called
                
    return capital

# ------------------------------------------------------------------------------
# 5. RUN EXPERIMENT
# ------------------------------------------------------------------------------
if not ml_scores.empty:
    
    cohorts = {
        "10 Years Ago": (datetime.today() - timedelta(days=365*10)).strftime('%Y-%m-%d'),
        "5 Years Ago": (datetime.today() - timedelta(days=365*5)).strftime('%Y-%m-%d'),
        "3 Years Ago": (datetime.today() - timedelta(days=365*3)).strftime('%Y-%m-%d'),
        "1 Year Ago": (datetime.today() - timedelta(days=365)).strftime('%Y-%m-%d'),
        "6 Months Ago": (datetime.today() - timedelta(days=180)).strftime('%Y-%m-%d')
    }

    print("\n" + "="*105)
    print(f"📈 FULL TIMELINE ML BACKTEST (Starting: $5k | Leverage: 2.0x Margin)")
    print(f"Warning: Years prior to {split_date.strftime('%Y-%m-%d')} are In-Sample (Model knew the data).")
    print("="*105)
    print(f"{'Timeframe':<15} | {'Daily Check':<18} | {'Weekly Check':<18} | {'Monthly Check':<18} | {'Yearly Check':<18}")
    print("-" * 105)

    for label, d_str in cohorts.items():
        res_d = run_ml_sim(d_str, 'D')
        res_w = run_ml_sim(d_str, 'W')
        res_m = run_ml_sim(d_str, 'M')
        res_y = run_ml_sim(d_str, 'Y')
        
        str_d = f"${res_d:,.2f}" if res_d > 0 else "WIPED OUT ($0)"
        str_w = f"${res_w:,.2f}" if res_w > 0 else "WIPED OUT ($0)"
        str_m = f"${res_m:,.2f}" if res_m > 0 else "WIPED OUT ($0)"
        str_y = f"${res_y:,.2f}" if res_y > 0 else "WIPED OUT ($0)"
        
        vals = [res_d, res_w, res_m, res_y]
        max_val = max(vals)
        
        if res_d == max_val and res_d > 0: str_d = f"🏆 {str_d}"
        if res_w == max_val and res_w > 0: str_w = f"🏆 {str_w}"
        if res_m == max_val and res_m > 0: str_m = f"🏆 {str_m}"
        if res_y == max_val and res_y > 0: str_y = f"🏆 {str_y}"
        
        print(f"{label:<15} | {str_d:<18} | {str_w:<18} | {str_m:<18} | {str_y:<18}")
            
    print("-" * 105)


Step 1: Authenticating with Alpaca & Downloading Data...
Step 2: Engineering Causal Features for the entire timeline...
Step 3: Training Model and Scoring the entire historical timeline...
Step 4: Running the Backtest across Frequencies...

📈 FULL TIMELINE ML BACKTEST (Starting: $5k | Leverage: 2.0x Margin)
Timeframe       | Daily Check        | Weekly Check       | Monthly Check      | Yearly Check      
---------------------------------------------------------------------------------------------------------
10 Years Ago    | 🏆 $18,835.74       | $8,211.93          | $13,423.34         | WIPED OUT ($0)    
5 Years Ago     | 🏆 $18,835.74       | $8,211.93          | $13,423.34         | WIPED OUT ($0)    
3 Years Ago     | $29,125.39         | $38,031.62         | 🏆 $69,765.21       | $11,013.51        
1 Year Ago      | $3,002.63          | $2,211.76          | 🏆 $4,535.01        | $444.13           
6 Months Ago    | 🏆 $4,360.97        | $1,744.63          | $1,380.41          | $1,4